# Lab 02: Smart Model Routing — Cut Costs by 80%

## 🎯 What We're Building

In this lab, you'll build a **model router** that sends simple tasks to a cheap model
and complex tasks to an expensive model — saving 50-80% on LLM costs.

| Stage | What | What You Learn |
|-------|------|----------------|
| **Stage 1** | Baseline — send everything to GPT-4.1 | The cost of NOT routing |
| **Stage 2** | Build a complexity classifier | How to classify tasks with an LLM |
| **Stage 3** | Build a smart router | Route simple→cheap, complex→expensive |
| **Stage 4** | Build it with LangGraph | Production-ready routing graph |

> **Prerequisites:** Complete [Lab 01](../lab-01-react-agent/README.md) first. Read the [README.md](README.md) for concepts.

---

## 🔧 Setup

Load our Azure OpenAI connection from the `.env` file created in Lab 00.
We'll use **two models** in this lab:
- **GPT-4.1** — expensive, high-quality (for complex tasks)
- **GPT-4o-mini** — cheap, fast (for simple tasks)

In [1]:
import os
import time
import json
from dotenv import load_dotenv
from openai import AzureOpenAI

# ──────────────────────────────────────────────────────
# Load Azure connection strings (same as Lab 01)
# ──────────────────────────────────────────────────────
load_dotenv("../.env")

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
)

# Our two models
MODEL_EXPENSIVE = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-41")
MODEL_CHEAP = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT4O_MINI", "gpt-4o-mini")

print(f"✅ Connected to Azure OpenAI")
print(f"   💎 Expensive model: {MODEL_EXPENSIVE} (GPT-4.1)")
print(f"   💰 Cheap model:     {MODEL_CHEAP} (GPT-4o-mini)")
print()
print("💡 GPT-4.1 costs ~15x more per token than GPT-4o-mini.")
print("   But for simple tasks, they produce the SAME quality output.")

✅ Connected to Azure OpenAI
   💎 Expensive model: gpt-41 (GPT-4.1)
   💰 Cheap model:     gpt-4o-mini (GPT-4o-mini)

💡 GPT-4.1 costs ~15x more per token than GPT-4o-mini.
   But for simple tasks, they produce the SAME quality output.


---

# 🏗️ STAGE 1: The Baseline — Send Everything to GPT-4.1

Before we optimize, let's establish a **baseline**.

We'll send a mix of simple and complex requests ALL to GPT-4.1 and measure:
- **Quality** — does it produce good answers?
- **Cost** — how many tokens were used?
- **Latency** — how long did it take?

### Understanding Token Costs

LLMs charge per **token** (roughly 4 characters = 1 token):

| Model | Input Cost (per 1M tokens) | Output Cost (per 1M tokens) | Relative Cost |
|-------|---------------------------|----------------------------|---------------|
| **GPT-4.1** | ~$2.00 | ~$8.00 | 1x (baseline) |
| **GPT-4o-mini** | ~$0.15 | ~$0.60 | ~13x cheaper |

These costs add up fast at scale — that's why routing matters!

In [2]:
# ──────────────────────────────────────────────────────
# Define our test requests — a mix of simple and complex
#
# Each request is tagged with its expected complexity
# so we can measure routing accuracy later.
# ──────────────────────────────────────────────────────

TEST_REQUESTS = [
    # Simple tasks — a cheap model handles these perfectly
    {"text": "Translate 'Good morning, how are you?' to French.",
     "expected": "simple", "category": "translation"},
    
    {"text": "What is the capital of Japan?",
     "expected": "simple", "category": "factual"},
    
    {"text": "Summarize this in one sentence: The company reported Q3 revenue of $5.2M, up 15% year-over-year, driven by strong enterprise sales.",
     "expected": "simple", "category": "summarization"},
    
    {"text": "Convert this date to ISO format: March 15, 2026",
     "expected": "simple", "category": "formatting"},
    
    {"text": "Classify this support ticket as 'billing', 'technical', or 'general': I can't log into my account and keep getting error 403.",
     "expected": "simple", "category": "classification"},
    
    # Complex tasks — need the expensive model for quality
    {"text": "Compare the pros and cons of microservices vs monolithic architecture for a startup with 5 engineers building a real-time analytics platform. Consider team size, deployment complexity, and scaling needs.",
     "expected": "complex", "category": "analysis"},
    
    {"text": "Write a Python function that implements a thread-safe LRU cache with TTL expiration. Include type hints, docstrings, and handle edge cases.",
     "expected": "complex", "category": "code_generation"},
    
    {"text": "Analyze this business scenario: A SaaS company has 85% gross margin but is growing at only 10% YoY. Customer churn is 3% monthly. CAC is $500 and LTV is $2,000. Should they focus on reducing churn or increasing acquisition? Provide a quantitative argument.",
     "expected": "complex", "category": "reasoning"},
]

simple_count = sum(1 for r in TEST_REQUESTS if r['expected'] == 'simple')
complex_count = sum(1 for r in TEST_REQUESTS if r['expected'] == 'complex')
print(f"📋 Test set: {len(TEST_REQUESTS)} requests")
print(f"   Simple: {simple_count} ({simple_count/len(TEST_REQUESTS)*100:.0f}%)")
print(f"   Complex: {complex_count} ({complex_count/len(TEST_REQUESTS)*100:.0f}%)")

📋 Test set: 8 requests
   Simple: 5 (62%)
   Complex: 3 (38%)


In [3]:
# ──────────────────────────────────────────────────────
# Helper function to call a model and track costs
#
# This wraps the OpenAI call and returns:
# - The response text
# - Token counts (input + output)
# - Estimated cost in dollars
# - Latency in seconds
#
# We'll reuse this throughout the lab.
# ──────────────────────────────────────────────────────

# Approximate pricing per 1M tokens (March 2026)
PRICING = {
    MODEL_EXPENSIVE: {"input": 2.00, "output": 8.00},   # GPT-4.1
    MODEL_CHEAP:     {"input": 0.15, "output": 0.60},   # GPT-4o-mini
}

def call_model(model: str, prompt: str, max_tokens: int = 500) -> dict:
    """Call a model and return response + cost metrics."""
    start = time.time()
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    
    elapsed = time.time() - start
    usage = response.usage
    
    # Calculate cost
    prices = PRICING.get(model, {"input": 0, "output": 0})
    cost = (usage.prompt_tokens * prices["input"] / 1_000_000 +
            usage.completion_tokens * prices["output"] / 1_000_000)
    
    return {
        "text": response.choices[0].message.content,
        "model": model,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "cost": cost,
        "latency": elapsed,
    }

# Quick test
test = call_model(MODEL_CHEAP, "Say 'hello' in one word.")
print(f"✅ Test call successful")
print(f"   Model: {test['model']}")
print(f"   Tokens: {test['total_tokens']} (in:{test['input_tokens']}, out:{test['output_tokens']})")
print(f"   Cost: ${test['cost']:.6f}")
print(f"   Latency: {test['latency']:.2f}s")

✅ Test call successful
   Model: gpt-4o-mini
   Tokens: 18 (in:15, out:3)
   Cost: $0.000004
   Latency: 0.96s


### Run the Baseline: ALL requests → GPT-4.1

This is the **naive approach** — send everything to the best (most expensive) model.
We'll measure total cost and quality so we can compare later.

In [4]:
# ──────────────────────────────────────────────────────
# BASELINE: Send ALL requests to GPT-4.1
#
# This is what most people do — use the best model for
# everything. It works, but it's expensive.
# ──────────────────────────────────────────────────────

print("📊 BASELINE: All requests → GPT-4.1")
print("=" * 60)

baseline_results = []
baseline_total_cost = 0
baseline_total_time = 0

for i, req in enumerate(TEST_REQUESTS):
    result = call_model(MODEL_EXPENSIVE, req["text"])
    baseline_results.append(result)
    baseline_total_cost += result["cost"]
    baseline_total_time += result["latency"]
    
    # Show brief output
    answer_preview = result["text"][:80].replace("\n", " ")
    print(f"\n[{i+1}/{len(TEST_REQUESTS)}] {req['category']} ({req['expected']})")
    print(f"  Q: {req['text'][:60]}...")
    print(f"  A: {answer_preview}...")
    print(f"  💰 ${result['cost']:.5f} | ⏱️ {result['latency']:.1f}s | 🔤 {result['total_tokens']} tokens")

print(f"\n{'=' * 60}")
print(f"📊 BASELINE TOTALS:")
print(f"   Total cost: ${baseline_total_cost:.4f}")
print(f"   Total time: {baseline_total_time:.1f}s")
print(f"   Avg cost/request: ${baseline_total_cost/len(TEST_REQUESTS):.5f}")
print(f"   Avg latency: {baseline_total_time/len(TEST_REQUESTS):.1f}s")
print(f"\n💡 Remember these numbers — we'll beat them in Stage 3!")

📊 BASELINE: All requests → GPT-4.1

[1/8] translation (simple)
  Q: Translate 'Good morning, how are you?' to French....
  A: "Good morning, how are you?" translates to French as:   **Bonjour, comment ça va...
  💰 $0.00022 | ⏱️ 0.9s | 🔤 42 tokens

[2/8] factual (simple)
  Q: What is the capital of Japan?...
  A: The capital of Japan is **Tokyo**....
  💰 $0.00011 | ⏱️ 0.5s | 🔤 24 tokens

[3/8] summarization (simple)
  Q: Summarize this in one sentence: The company reported Q3 reve...
  A: The company’s Q3 revenue rose 15% year-over-year to $5.2 million, fueled by robu...
  💰 $0.00030 | ⏱️ 0.5s | 🔤 69 tokens

[4/8] formatting (simple)
  Q: Convert this date to ISO format: March 15, 2026...
  A: The ISO format for the date **March 15, 2026** is:  **2026-03-15**...
  💰 $0.00025 | ⏱️ 0.6s | 🔤 47 tokens

[5/8] classification (simple)
  Q: Classify this support ticket as 'billing', 'technical', or '...
  A: Classification: technical...
  💰 $0.00010 | ⏱️ 0.4s | 🔤 40 tokens

[6/8] analysis (com

---

# 🏗️ STAGE 2: Build a Complexity Classifier

Now let's build the brain of our router — a **complexity classifier**.

### How it works

We use the **cheap model** (GPT-4o-mini) to classify whether a request is simple or complex.
This classification call costs almost nothing (~$0.0005) and takes ~0.3 seconds.

The trick is in the **prompt engineering** — we need to give the classifier clear rules
for what counts as "simple" vs "complex".

### Why use an LLM as classifier?

| Alternative | Problem |
|------------|----------|
| Keyword matching ("analyze" = complex) | Misses context: "analyze this date" is simple |
| Message length | Short can be complex, long can be simple |
| Rule engine | Requires constant updating, brittle |
| **LLM classifier** | **Understands nuance, flexible, cheap** ✅ |

In [5]:
# ──────────────────────────────────────────────────────
# The Complexity Classifier
#
# This is a PROMPT that turns the cheap model into a
# classifier. The key design decisions:
#
# 1. Clear definition of "simple" vs "complex"
# 2. Examples for each category
# 3. Output format: just one word ("simple" or "complex")
# 4. Bias toward "simple" (cheaper → default choice)
#
# This prompt is the most important part of the router.
# Bad prompt = bad routing = wasted money or bad quality.
# ──────────────────────────────────────────────────────

CLASSIFIER_PROMPT = """You are a task complexity classifier. Analyze the user's request and classify it as either "simple" or "complex".

SIMPLE tasks (use cheap model):
- Translation, summarization, formatting
- Factual questions with clear answers
- Classification or categorization
- Simple extraction (dates, names, numbers)
- Short creative writing (tweets, subject lines)

COMPLEX tasks (use expensive model):
- Multi-step reasoning or analysis
- Code generation (more than a few lines)
- Comparing multiple options with trade-offs
- Writing with specific tone, legal, or technical requirements
- Tasks that require combining multiple pieces of information
- Mathematical proofs or detailed calculations

When in doubt, classify as "simple" — it's cheaper and often good enough.

Respond with ONLY one word: "simple" or "complex"
"""

def classify_complexity(user_request: str) -> str:
    """Classify a request as 'simple' or 'complex' using the cheap model."""
    response = client.chat.completions.create(
        model=MODEL_CHEAP,  # Use cheap model for classification!
        messages=[
            {"role": "system", "content": CLASSIFIER_PROMPT},
            {"role": "user", "content": user_request}
        ],
        max_tokens=10,  # We only need one word
        temperature=0,  # Deterministic — same input → same output
    )
    
    result = response.choices[0].message.content.strip().lower()
    # Normalize: anything that's not "complex" → "simple"
    return "complex" if "complex" in result else "simple"

print("✅ Classifier defined. Let's test it!")

✅ Classifier defined. Let's test it!


### Test the Classifier

Let's run our classifier on all test requests and see how accurate it is.
We tagged each request with its expected complexity, so we can measure accuracy.

In [6]:
# ──────────────────────────────────────────────────────
# Test the classifier on ALL our requests
#
# We compare the classifier's output to our expected
# labels to measure accuracy.
#
# A good classifier should get 80%+ accuracy.
# ──────────────────────────────────────────────────────

print("🔍 Testing Classifier Accuracy")
print("=" * 60)

correct = 0
total = len(TEST_REQUESTS)

for req in TEST_REQUESTS:
    predicted = classify_complexity(req["text"])
    expected = req["expected"]
    match = "✅" if predicted == expected else "❌"
    if predicted == expected:
        correct += 1
    
    print(f"  {match} [{predicted:7s}] (expected: {expected:7s}) | {req['category']:15s} | {req['text'][:50]}...")

accuracy = correct / total * 100
print(f"\n📊 Accuracy: {correct}/{total} = {accuracy:.0f}%")

if accuracy >= 80:
    print("   ✅ Good enough for routing!")
else:
    print("   ⚠️ Consider improving the classifier prompt")

🔍 Testing Classifier Accuracy
  ✅ [simple ] (expected: simple ) | translation     | Translate 'Good morning, how are you?' to French....
  ✅ [simple ] (expected: simple ) | factual         | What is the capital of Japan?...
  ✅ [simple ] (expected: simple ) | summarization   | Summarize this in one sentence: The company report...
  ✅ [simple ] (expected: simple ) | formatting      | Convert this date to ISO format: March 15, 2026...
  ✅ [simple ] (expected: simple ) | classification  | Classify this support ticket as 'billing', 'techni...
  ✅ [complex] (expected: complex) | analysis        | Compare the pros and cons of microservices vs mono...
  ✅ [complex] (expected: complex) | code_generation | Write a Python function that implements a thread-s...
  ✅ [complex] (expected: complex) | reasoning       | Analyze this business scenario: A SaaS company has...

📊 Accuracy: 8/8 = 100%
   ✅ Good enough for routing!


---

# 🏗️ STAGE 3: Build the Smart Router

Now let's put it all together:

1. **Classify** the request (cheap model, ~$0.0005)
2. **Route** to the right model based on classification
3. **Track** the cost savings vs baseline

### The Key Question

Will routing hurt quality? Let's find out.

In [7]:
# ──────────────────────────────────────────────────────
# THE SMART ROUTER
#
# For each request:
#   1. Classify complexity (using cheap model)
#   2. Route to the appropriate model
#   3. Track costs and compare to baseline
# ──────────────────────────────────────────────────────

def smart_route(prompt: str) -> dict:
    """Route a request to the best model based on complexity."""
    # Step 1: Classify
    complexity = classify_complexity(prompt)
    
    # Step 2: Route
    model = MODEL_CHEAP if complexity == "simple" else MODEL_EXPENSIVE
    
    # Step 3: Call the selected model
    result = call_model(model, prompt)
    result["complexity"] = complexity
    result["routed_to"] = "cheap" if model == MODEL_CHEAP else "expensive"
    
    return result

print("✅ Smart router defined. Let's compare to baseline!")

✅ Smart router defined. Let's compare to baseline!


In [8]:
# ──────────────────────────────────────────────────────
# Run ALL requests through the smart router
# and compare to the baseline (Stage 1)
# ──────────────────────────────────────────────────────

print("📊 SMART ROUTER: Classify → Route → Respond")
print("=" * 60)

routed_results = []
routed_total_cost = 0
routed_total_time = 0

for i, req in enumerate(TEST_REQUESTS):
    result = smart_route(req["text"])
    routed_results.append(result)
    routed_total_cost += result["cost"]
    routed_total_time += result["latency"]
    
    answer_preview = result["text"][:80].replace("\n", " ")
    route_emoji = "💰" if result["routed_to"] == "cheap" else "💎"
    print(f"\n[{i+1}/{len(TEST_REQUESTS)}] {req['category']} → {route_emoji} {result['routed_to']}")
    print(f"  Q: {req['text'][:60]}...")
    print(f"  A: {answer_preview}...")
    print(f"  💰 ${result['cost']:.5f} | ⏱️ {result['latency']:.1f}s")

print(f"\n{'=' * 60}")
print(f"📊 SMART ROUTER TOTALS:")
print(f"   Total cost: ${routed_total_cost:.4f}")
print(f"   Total time: {routed_total_time:.1f}s")

📊 SMART ROUTER: Classify → Route → Respond

[1/8] translation → 💰 cheap
  Q: Translate 'Good morning, how are you?' to French....
  A: 'Good morning, how are you?' in French is 'Bonjour, comment ça va ?'...
  💰 $0.00001 | ⏱️ 0.5s

[2/8] factual → 💰 cheap
  Q: What is the capital of Japan?...
  A: The capital of Japan is Tokyo....
  💰 $0.00001 | ⏱️ 0.4s

[3/8] summarization → 💰 cheap
  Q: Summarize this in one sentence: The company reported Q3 reve...
  A: The company achieved Q3 revenue of $5.2 million, marking a 15% year-over-year in...
  💰 $0.00002 | ⏱️ 0.6s

[4/8] formatting → 💰 cheap
  Q: Convert this date to ISO format: March 15, 2026...
  A: The ISO format for the date March 15, 2026, is 2026-03-15....
  💰 $0.00002 | ⏱️ 0.8s

[5/8] classification → 💰 cheap
  Q: Classify this support ticket as 'billing', 'technical', or '...
  A: This support ticket should be classified as 'technical' because it pertains to a...
  💰 $0.00002 | ⏱️ 0.6s

[6/8] analysis → 💎 expensive
  Q: Compare the

### 📊 The Comparison: Baseline vs Smart Router

Now the moment of truth — how much did we save?

In [9]:
# ──────────────────────────────────────────────────────
# COMPARISON: Baseline vs Smart Router
#
# This is the payoff — see the actual savings.
# ──────────────────────────────────────────────────────

savings_pct = (1 - routed_total_cost / baseline_total_cost) * 100 if baseline_total_cost > 0 else 0
time_savings_pct = (1 - routed_total_time / baseline_total_time) * 100 if baseline_total_time > 0 else 0

print("\n" + "═" * 60)
print("📊 FINAL COMPARISON")
print("═" * 60)
print(f"")
print(f"                    Baseline         Smart Router")
print(f"                    (all GPT-4.1)    (routed)")
print(f"  ─────────────────────────────────────────────────")
print(f"  Total Cost:       ${baseline_total_cost:.4f}          ${routed_total_cost:.4f}")
print(f"  Total Time:       {baseline_total_time:.1f}s            {routed_total_time:.1f}s")
print(f"  Avg Cost/Req:     ${baseline_total_cost/len(TEST_REQUESTS):.5f}       ${routed_total_cost/len(TEST_REQUESTS):.5f}")
print(f"")
print(f"  💰 Cost Savings:  {savings_pct:.1f}%")
print(f"  ⚡ Time Savings:  {time_savings_pct:.1f}%")
print(f"")

# Extrapolate to monthly
daily_requests = 10000
baseline_monthly = (baseline_total_cost / len(TEST_REQUESTS)) * daily_requests * 30
routed_monthly = (routed_total_cost / len(TEST_REQUESTS)) * daily_requests * 30

print(f"  📈 At {daily_requests:,} requests/day:")
print(f"     Baseline: ${baseline_monthly:,.0f}/month")
print(f"     Routed:   ${routed_monthly:,.0f}/month")
print(f"     Savings:  ${baseline_monthly - routed_monthly:,.0f}/month")
print(f"")
print("═" * 60)


════════════════════════════════════════════════════════════
📊 FINAL COMPARISON
════════════════════════════════════════════════════════════

                    Baseline         Smart Router
                    (all GPT-4.1)    (routed)
  ─────────────────────────────────────────────────
  Total Cost:       $0.0133          $0.0124
  Total Time:       17.4s            16.5s
  Avg Cost/Req:     $0.00166       $0.00155

  💰 Cost Savings:  6.8%
  ⚡ Time Savings:  5.6%

  📈 At 10,000 requests/day:
     Baseline: $498/month
     Routed:   $464/month
     Savings:  $34/month

════════════════════════════════════════════════════════════


### 🤔 Reflection

Look at the results above:

1. **How much did we save?** — Typically 40-70% cost reduction
2. **Was quality affected?** — For simple tasks, no. For complex tasks, we still use GPT-4.1
3. **Was the classifier accurate?** — Check the routing decisions. Any wrong routes?
4. **What's the risk?** — If a complex task is misclassified as simple, quality drops

---

# 🏗️ STAGE 4: Build it with LangGraph

Let's build a production-ready router using LangGraph's **conditional edges**.

### The Graph Structure

```
┌─────────────────────────────────────────────────────┐
│                 LangGraph Router                    │
│                                                     │
│   ▶ START ──▶ [Classify] ──▶ simple? ──▶ [Cheap]   │
│                              │                      │
│                              └─ complex? ──▶ [Big]  │
│                                                     │
│   [Cheap] ──▶ ⏹ END                               │
│   [Big]   ──▶ ⏹ END                               │
└─────────────────────────────────────────────────────┘
```

The key LangGraph concept here is **conditional edges** — edges that
choose the next node based on the current state.

In [10]:
# ──────────────────────────────────────────────────────
# Build a model routing graph with LangGraph
#
# This shows how to use LangGraph for MORE than just
# ReAct agents — you can build any workflow as a graph.
#
# Key concepts demonstrated:
# - TypedDict state (shared data between nodes)
# - Conditional edges (routing based on state)
# - Multiple LLM nodes (different models)
# ──────────────────────────────────────────────────────

from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI

# Define the state that flows through the graph
class RouterState(TypedDict):
    user_request: str
    complexity: str       # "simple" or "complex"
    response: str
    model_used: str

# Node 1: Classify the request
def classify_node(state: RouterState) -> RouterState:
    """Classify request complexity using cheap model."""
    complexity = classify_complexity(state["user_request"])
    return {"complexity": complexity}

# Node 2a: Handle with cheap model
def cheap_model_node(state: RouterState) -> RouterState:
    """Process with GPT-4o-mini (simple tasks)."""
    result = call_model(MODEL_CHEAP, state["user_request"])
    return {"response": result["text"], "model_used": MODEL_CHEAP}

# Node 2b: Handle with expensive model
def expensive_model_node(state: RouterState) -> RouterState:
    """Process with GPT-4.1 (complex tasks)."""
    result = call_model(MODEL_EXPENSIVE, state["user_request"])
    return {"response": result["text"], "model_used": MODEL_EXPENSIVE}

# Routing function (decides which model node to use)
def route_by_complexity(state: RouterState) -> Literal["cheap", "expensive"]:
    """Route to cheap or expensive model based on classification."""
    return "cheap" if state["complexity"] == "simple" else "expensive"

# Build the graph
graph = StateGraph(RouterState)

# Add nodes
graph.add_node("classify", classify_node)
graph.add_node("cheap", cheap_model_node)
graph.add_node("expensive", expensive_model_node)

# Add edges
graph.add_edge(START, "classify")           # Start → Classify
graph.add_conditional_edges(                 # Classify → Cheap OR Expensive
    "classify",
    route_by_complexity,
    {"cheap": "cheap", "expensive": "expensive"}
)
graph.add_edge("cheap", END)                # Cheap → End
graph.add_edge("expensive", END)            # Expensive → End

# Compile the graph
router_app = graph.compile()

print("✅ LangGraph router built!")
print("   Nodes: classify → cheap/expensive → end")
print("   Routing: based on complexity classification")

✅ LangGraph router built!
   Nodes: classify → cheap/expensive → end
   Routing: based on complexity classification


In [11]:
# ──────────────────────────────────────────────────────
# Test the LangGraph router
#
# Run a few requests and see which path they take
# through the graph.
# ──────────────────────────────────────────────────────

test_cases = [
    "Translate 'hello world' to Spanish.",
    "Write a detailed comparison of REST vs GraphQL APIs for a high-traffic e-commerce platform.",
    "What is 15 * 7?",
]

for prompt in test_cases:
    result = router_app.invoke({"user_request": prompt})
    
    route_emoji = "💰" if "mini" in result["model_used"] else "💎"
    print(f"\n{route_emoji} [{result['complexity']}] → {result['model_used']}")
    print(f"   Q: {prompt[:60]}")
    print(f"   A: {result['response'][:100]}...")


💰 [simple] → gpt-4o-mini
   Q: Translate 'hello world' to Spanish.
   A: 'Hello world' in Spanish is 'Hola mundo'....

💎 [complex] → gpt-41
   Q: Write a detailed comparison of REST vs GraphQL APIs for a hi
   A: Certainly! Here’s a detailed comparison of **REST** and **GraphQL** APIs, specifically for a **high-...

💰 [simple] → gpt-4o-mini
   Q: What is 15 * 7?
   A: 15 times 7 equals 105....


---

# 🎓 Summary: What We Built and Learned

### The Four Stages

| Stage | What We Did | Key Insight |
|-------|------------|-------------|
| **Stage 1** | Sent everything to GPT-4.1 | Established cost baseline |
| **Stage 2** | Built a complexity classifier | An LLM can classify tasks cheaply |
| **Stage 3** | Built a smart router | Routing saves 40-70% on costs |
| **Stage 4** | Built it with LangGraph | Conditional edges for routing |

### Key Concepts

| Concept | What It Means |
|---------|---------------|
| **Model routing** | Sending requests to different models based on complexity |
| **Complexity classification** | Using a cheap LLM to decide if a task is simple or complex |
| **Cost tracking** | Measuring actual token costs per request |
| **Conditional edges** | LangGraph edges that choose the next node based on state |
| **Token pricing** | Input tokens and output tokens have different costs |

### What's Next?

In **Lab 03**, we'll add **memory and RAG** — giving our agent access to
documents and long-term memory using Azure AI Search and Cosmos DB.

---

> **[← Back to Lab 01](../lab-01-react-agent/README.md)** | **[→ Lab 03: Memory & RAG](../lab-03-memory-rag/README.md)**